In [1]:
import dill as pickle
from data_loaders import get_datasets, CATEGORICAL_FEATURES
from policy import RURAL_COVARIATE_LIST
import numpy as np
import itertools
from reporting import write_result

In [2]:
def load_policy(file_path):
    with open(file_path, 'rb') as f:
        t = pickle.load(f)
    return t

In [3]:
t = load_policy("policies/chitipa_binary_uncondtol=0.1.pickle")

In [4]:
fold1, fold2, features = get_datasets("chitipa", pool="central", covariates=RURAL_COVARIATE_LIST)

/zfs/gsb/intermediate-yens/rsahoo/poverty/gd/data_loaders.py:103: DtypeWarning: Columns (10,20,24,26,28,31,33,37,39,40,43,44,45,46,47,49,50,52,53,55,58,60,64,72,73,80,82,83,84,85,87,89,90,98,151,164,166,168,170,171,172,175,176,187,190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH_TO_DATA)
/zfs/gsb/intermediate-yens/rsahoo/poverty/gd/data_loaders.py:103: DtypeWarning: Columns (10,20,24,26,28,31,33,37,39,40,43,44,45,46,47,49,50,52,53,55,58,60,64,72,73,80,82,83,84,85,87,89,90,98,151,164,166,168,170,171,172,175,176,187,190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PATH_TO_DATA)


Index(['hh_f06_PERMANENT', 'hh_f06_SEMI-PERMANENT', 'hh_f06_TRADITIONAL',
       'hh_f07_BURNT BRICKS', 'hh_f07_COMPACTED EARTH(YAMDINDO)',
       'hh_f07_CONCRETE', 'hh_f07_GRASS', 'hh_f07_IRON SHEETS',
       'hh_f07_MUD (YOMATA)', 'hh_f07_MUD BRICK(UNFIRED)',
       'hh_f07_OTHER (SPECIFY)', 'hh_f07_WOOD', 'hh_f12_CHARCOAL',
       'hh_f12_COLLECTED FIREWOOD', 'hh_f12_CROP RESIDUE',
       'hh_f12_ELECTRICITY', 'hh_f12_GAS', 'hh_f12_OTHER (SPECIFY)',
       'hh_f12_PURCHASED FIREWOOD', 'hh_f12_SAW DUST', 'hh_f19_NO',
       'hh_f19_YES'],
      dtype='object')


In [10]:
def get_num_cats(cat_feat, features):
    bs = []
    for i, w in enumerate(features):
        if w.startswith(cat_feat):
            bs.append(i)
    return len(bs)

num_cats = [get_num_cats(cat_feat, features) for cat_feat in RURAL_COVARIATE_LIST]
print(num_cats)


[3, 9, 8, 2]


In [55]:
def map_one_hot_to_name(features, cats, one_hot):
    name_dic = {}

    idx = 0
    for i, feature in enumerate(features):
        if one_hot[i] == 1:
            name_dic[cats[idx]] = feature.removeprefix("{}_".format(cats[idx]))
            idx += 1
    return name_dic

def generate_one_hot_vectors(categories, subcategories, num_categories):
    indices = [np.eye(num) for num in num_categories]
    index_combinations = itertools.product(*indices)
    one_hot_vectors = []
    names = []
    for combo in index_combinations:
        vector = []
        for i in range(len(categories)):
            vector.extend(list(combo[i]))
        one_hot_vectors.append(vector)
        names.append(map_one_hot_to_name(subcategories, RURAL_COVARIATE_LIST, vector))
    return np.array(one_hot_vectors), names

def update_namedic_with_transfer_values(name_dics, transfers):
    for i, name_dic in enumerate(name_dics):
        tx = 0
        for tup in transfers[i]:
            tx += tup[0] * tup[1]

        name_dic["expected_transfer"] = tx
    return name_dics


def export_namedics_to_csv(filename, name_dics):
    for name_dic in name_dics:
        write_result("policy_tables/{}.csv".format(filename), name_dic)
    


In [56]:
one_hot_vectors, names = generate_one_hot_vectors(RURAL_COVARIATE_LIST, features, num_cats)

In [57]:
update_namedic_with_transfer_values(names, t(one_hot_vectors))

[{'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'CHARCOAL',
  'hh_f19': 'NO',
  'expected_transfer': 0.0},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'CHARCOAL',
  'hh_f19': 'YES',
  'expected_transfer': 0.0},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'COLLECTED FIREWOOD',
  'hh_f19': 'NO',
  'expected_transfer': 1.5333333333333332},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'COLLECTED FIREWOOD',
  'hh_f19': 'YES',
  'expected_transfer': 0.0},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'CROP RESIDUE',
  'hh_f19': 'NO',
  'expected_transfer': 1.5333333333333332},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'CROP RESIDUE',
  'hh_f19': 'YES',
  'expected_transfer': 0.0},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'ELECTRICITY',
  'hh_f19': 'NO',
  'expected_transfer': 0.0},
 {'hh_f06': 'PERMANENT',
  'hh_f07': 'BURNT BRICKS',
  'hh_f12': 'EL